# F1 正賽資料大表建置（2011 年以後）

這份 notebook 把 Ergast F1 資料集的 14 個 CSV（加上天氣 API）串成一張分析用的大表 `f1_master.csv`。

**最終大表的一列 =「一位車手」在「一場正賽」**，主鍵為 `(raceId, driverId)`。

最終欄位（順序與研究設計一致）：

| raceId | year | circuitId | driverId | constructorId | grid | quali_gap_pct | n_pit_stops | avg_pit_duration | lap_time_mean | lap_time_sd | drv_avg_finish_l5 | con_points_l5 | street_circuit | air_temp_mean | precip_mm | style_cluster | podium |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|

**整體流程**

| 步驟 | 內容 | 產出的欄位 |
|---|---|---|
| 0 | 環境設定、讀取 14 個檔案 | — |
| 1 | 建立主表（決定有哪些列） | raceId, year, circuitId, driverId, constructorId, grid, podium |
| 2 | 排位賽速度差距 | quali_gap_pct |
| 3 | 進站策略（前 5 場平均） | n_pit_stops, avg_pit_duration |
| 4 | 單圈時間風格（前 5 場平均） | lap_time_mean, lap_time_sd |
| 5 | 車手近期表現 | drv_avg_finish_l5 |
| 6 | 車隊近期表現 | con_points_l5 |
| 7 | 賽道類型（人工標記） | street_circuit |
| 8 | 天氣（Open-Meteo API） | air_temp_mean, precip_mm |
| 9 | 合併成大表、加上分群欄位佔位 | style_cluster（先留空） |
| 10 | 品質檢查：列數、重複、資料洩漏、缺值 | — |
| 11 | 輸出 CSV ＋ 欄位定義總表 | — |

**工具選擇**：資料串聯與「前 5 場」滾動計算用 SQL 寫（Python 內建的 `sqlite3`，不需另外安裝），字串解析、清理和呼叫 API 用 pandas。

## 步驟 0-1：環境設定

這個 cell 做三件事：

1. 載入套件：`pandas`（資料處理）、`numpy`（數值運算）、`sqlite3`（在記憶體中開一個 SQL 資料庫，用來做 join 和 window function）、`requests`（呼叫天氣 API）。
2. 設定全域參數，集中在這裡，之後要改只改這裡：
   - `DATA_DIR`：14 個 CSV 所在的資料夾。
   - `START_YEAR = 2011`：最終大表只保留 2011 年以後，因為 `pit_stops.csv` 從 2011 年才有資料。
   - `WINDOW = 5`：「前 5 場」的視窗大小。
   - `NA = ['\\N']`：Ergast 用 `\N` 表示缺值，讀檔時要告訴 pandas。
3. 建立一個空的記憶體 SQL 資料庫 `con`。

In [1]:
import sqlite3
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests

DATA_DIR   = Path('.')      # 14 個 CSV 所在資料夾（請依你的電腦修改）
OUT_DIR    = Path('.')      # 輸出位置
START_YEAR = 2011           # 最終大表的起始年份
WINDOW     = 5              # 「前 N 場」的 N
NA         = ['\\N']        # Ergast 的缺值記號

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)

con = sqlite3.connect(':memory:')   # 記憶體中的 SQL 資料庫，關掉 notebook 就消失
print('pandas', pd.__version__, '| sqlite', sqlite3.sqlite_version)

pandas 3.0.5 | sqlite 3.45.1


## 步驟 0-2：讀取 14 個檔案

把 14 個 CSV 全部讀進來，存在字典 `raw` 裡（key 是檔名，例如 `raw['results']`），並列出每個檔案的列數、欄數。

讀檔時指定 `na_values=['\\N']`，所以原本的 `\N` 會變成 pandas 的缺值 `NaN`，數字欄位才能被正確辨識為數字。

In [2]:
FILES = ['circuits', 'constructor_results', 'constructor_standings', 'constructors',
         'driver_standings', 'drivers', 'lap_times', 'pit_stops', 'qualifying',
         'races', 'results', 'seasons', 'sprint_results', 'status']

raw = {name: pd.read_csv(DATA_DIR / f'{name}.csv', na_values=NA) for name in FILES}

pd.DataFrame({'列數': {k: v.shape[0] for k, v in raw.items()},
              '欄數': {k: v.shape[1] for k, v in raw.items()}})

,列數,欄數
circuits,77,9
constructor_results,12625,5
constructor_standings,13391,7
constructors,212,5
driver_standings,34863,7
drivers,861,9
lap_times,589081,6
pit_stops,11371,7
qualifying,10494,9
races,1125,18


## 步驟 0-3：決定要用哪些表

14 張表裡，只有 7 張是建這張大表需要的。下面列出每張表的用途，或為什麼不用：

| 檔案 | 是否使用 | 用途／不使用的原因 |
|---|---|---|
| `results` | ✅ | 主表來源：每位車手每場正賽一列，有 grid、名次、積分 |
| `races` | ✅ | 比賽日期（排序「前 5 場」用）、年份、賽道、開賽時間（天氣用） |
| `circuits` | ✅ | 賽道經緯度（天氣 API 用）、賽道代碼（街道賽標記用） |
| `qualifying` | ✅ | Q1/Q2/Q3 圈速 → `quali_gap_pct` |
| `pit_stops` | ✅ | 進站次數與時間 → `n_pit_stops`, `avg_pit_duration`；也用來排除進出站圈 |
| `lap_times` | ✅ | 每圈時間 → `lap_time_mean`, `lap_time_sd` |
| `constructors` | ✅（僅檢查用） | 車隊名稱，用來說明改名問題，不進大表 |
| `drivers` | ❌ | 只有姓名、國籍、生日，研究問題用不到（之後想加年齡可以再接） |
| `constructor_results` | ❌ | 和 `results` 按車隊加總的積分相同，用 `results` 即可，避免兩個來源不一致 |
| `constructor_standings` / `driver_standings` | ❌ | 累積積分榜。若要用，必須取「上一場賽後」的值，否則會洩漏當場結果；本表已用前 5 場平均表示近期表現 |
| `sprint_results` | ❌ | 衝刺賽。研究對象是正賽，且 2021 年後才有 |
| `status` | ❌ | 完賽狀態文字（引擎故障、碰撞等）。判斷是否完賽直接用 `results.position` 是否為空即可 |
| `seasons` | ❌ | 只有年份和網址 |

接著把會用到的表放進 SQL 資料庫，之後就可以用 SQL 查詢它們。

In [3]:
for name in ['results', 'races', 'circuits']:
    raw[name].to_sql(name, con, index=False)

# 確認放進去了
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con)

,name
0,results
1,races
2,circuits


## 步驟 1：建立主表 `base`

主表決定最終大表「有哪些列」。來源是 `results JOIN races`，每位車手每場正賽一列。

這裡**先保留所有年份**，最後才篩出 2011 年以後。原因：2011 年第一場比賽的「前 5 場平均」需要 2010 年的比賽資料。如果太早把 2010 年刪掉，2011 年初的列都會是缺值。

這個步驟產出的欄位：

- `raceId`, `year`, `circuitId`, `date`：比賽識別資訊。`date` 是後面「依時間排序」的依據。⚠️ 不能用 `raceId` 排序，因為 Ergast 的 raceId 不是按時間編號的（例如 2009 年的 raceId 是 1–17，比 1950 年代還小）。
- `driverId`, `constructorId`：車手、車隊。
- `grid`：起跑位置。`0` 代表從維修區起跑（pit lane start），這裡保留原值，後面的缺值檢查會統計有幾筆。
- `position`：正式名次，未被列入正式名次（退賽、取消資格）時為空。後面判斷「是否完賽」會用到。
- `positionOrder`：最終排序。所有人都有值，包含退賽者（排在完賽者之後）。用來算平均名次。
- `points`：該場正賽得分。
- `podium`：**目標變數**。`positionOrder <= 3` 為 1，否則為 0。

In [4]:
base = pd.read_sql('''
    SELECT  r.raceId, ra.year, ra.circuitId, ra.date,
            r.driverId, r.constructorId,
            r.grid, r.position, r.positionOrder, r.points,
            CASE WHEN r.positionOrder <= 3 THEN 1 ELSE 0 END AS podium
    FROM results r
    JOIN races   ra ON ra.raceId = r.raceId
    ORDER BY ra.date, r.raceId, r.positionOrder
''', con)

# 檢查主鍵 (raceId, driverId) 在 2011 年以後是否唯一
dup = base[base.year >= START_YEAR].duplicated(['raceId', 'driverId']).sum()
print('全部年份列數:', len(base))
print(f'{START_YEAR} 年以後列數:', (base.year >= START_YEAR).sum(), '| 重複的 (raceId, driverId):', dup)

base.to_sql('base', con, index=False)
base[base.year >= START_YEAR].head()

全部年份列數: 26759
2011 年以後列數: 5980 | 重複的 (raceId, driverId): 0


,raceId,year,circuitId,date,driverId,constructorId,grid,position,positionOrder,points,podium
20779,841,2011,1,2011-03-27,20,9,1,1.0,1,25.0,1
20780,841,2011,1,2011-03-27,1,1,2,2.0,2,18.0,1
20781,841,2011,1,2011-03-27,808,4,6,3.0,3,15.0,1
20782,841,2011,1,2011-03-27,4,6,5,4.0,4,12.0,0
20783,841,2011,1,2011-03-27,17,9,3,5.0,5,10.0,0


## 步驟 2：`quali_gap_pct`（排位賽速度差距）

**定義**：該車手在排位賽中最快的一圈，比該場最快排位圈慢多少百分比。

$$\text{quali\_gap\_pct} = \frac{\text{車手最佳圈} - \text{該場最快圈}}{\text{該場最快圈}} \times 100$$

計算方式：

1. `qualifying` 的 `q1`, `q2`, `q3` 是 `"1:23.456"` 格式的字串，先用 `to_seconds()` 轉成秒數。
2. 每位車手的最佳圈 = Q1、Q2、Q3 三者中的最小值。Q1 就被淘汰的車手只有 Q1 成績。
3. 該場最快圈 = 所有車手最佳圈中的最小值，通常就是桿位者的 Q3 成績。這裡用全場最小值而非桿位者的成績，確保差距不會出現負數（偶爾有人 Q2 跑得比桿位者 Q3 還快）。
4. 沒有任何有效圈速的車手（例如排位賽撞車、沒有參加）為缺值。

⚠️ **方法上的限制**：Q1 就淘汰的車手只有 Q1 成績，卻要和 Q3 的桿位圈比。

- 一般情況下，Q3 的賽道比較快（膠多），差距只會被些微高估。
- **天氣變化時問題很嚴重**：例如 2011 比利時、2013 澳洲、2023 荷蘭，Q1 下雨、Q3 轉乾（或反過來），Q1 淘汰者的差距會到 10%–45%，這反映的是天氣而不是速度。

所以這裡提供參數 `QUALI_METHOD` 讓你選擇：

- `'best'`（預設，符合你的原始定義）：車手最佳圈 vs 全場最快圈。
- `'q1'`：只用 Q1 比較。所有車手都跑過 Q1，而且是同一時段、同樣的天氣，比較公平；缺點是 Q1 不是頂尖車手的極限圈速。

排位賽在正賽之前，所以這個欄位沒有資料洩漏的問題。

In [5]:
def to_seconds(s):
    '''把 "1:23.456" 或 "83.456" 轉成秒數；空值或格式錯誤回傳 NaN'''
    if pd.isna(s):
        return np.nan
    s = str(s).strip()
    try:
        if ':' in s:
            m, sec = s.split(':')
            return int(m) * 60 + float(sec)
        return float(s)
    except ValueError:
        return np.nan

QUALI_METHOD = 'best'   # 'best'：最佳圈 vs 全場最快圈；'q1'：只比 Q1

q = raw['qualifying'][['raceId', 'driverId', 'q1', 'q2', 'q3']].copy()
for c in ['q1', 'q2', 'q3']:
    q[c + '_s'] = q[c].map(to_seconds)

if QUALI_METHOD == 'best':
    q['best_s'] = q[['q1_s', 'q2_s', 'q3_s']].min(axis=1)       # 車手最佳圈（秒）
else:
    q['best_s'] = q['q1_s']                                     # 只用 Q1
q['fastest_s'] = q.groupby('raceId')['best_s'].transform('min') # 該場最快圈（秒）
q['quali_gap_pct'] = (q['best_s'] - q['fastest_s']) / q['fastest_s'] * 100

# 同一場同一車手理論上只有一列；保險起見取最小值
feat_quali = q.groupby(['raceId', 'driverId'], as_index=False)['quali_gap_pct'].min()
feat_quali.to_sql('feat_quali', con, index=False)

print(feat_quali['quali_gap_pct'].describe().round(3))
q[['raceId', 'driverId', 'q1', 'q2', 'q3', 'best_s', 'fastest_s', 'quali_gap_pct']].tail()

count    10338.000
mean         2.490
std          9.411
min          0.000
25%          0.907
50%          1.841
75%          3.071
max        922.862
Name: quali_gap_pct, dtype: float64


,raceId,driverId,q1,q2,q3,best_s,fastest_s,quali_gap_pct
10489,1144,848,1:23.821,NaN,NaN,83.821,82.595,1.484351
10490,1144,855,1:23.880,NaN,NaN,83.880,82.595,1.555784
10491,1144,1,1:23.887,NaN,NaN,83.887,82.595,1.564259
10492,1144,861,1:23.912,NaN,NaN,83.912,82.595,1.594528
10493,1144,862,1:24.105,NaN,NaN,84.105,82.595,1.828198


## 步驟 3-1：清理進站資料

`pit_stops.milliseconds` 是車子從進入維修區到離開維修區的總時間（包含在維修區內行駛的時間，不只是換胎那幾秒）。

需要排除的異常值：

- **紅旗期間的進站**：紅旗時車子停在維修區等待重新開賽，紀錄可能長達數十分鐘（例如 2024 摩納哥、2023 澳洲）。這不是策略性進站。
- **修車等特殊停留**。

清理規則（兩個條件都要符合才保留）：

1. 絕對門檻：`milliseconds < 60,000`（60 秒）。
2. 相對門檻：不超過「該場有效進站時間中位數」的 1.5 倍。這能排除像 2021 艾米利亞-羅馬涅那種紅旗停留只有 50 多秒的情況。不同賽道維修區長度不同（約 17–35 秒），所以要用該場自己的中位數當基準。

下面印出清理前後的筆數，以及被排除最多的比賽。

In [6]:
pit = raw['pit_stops'][['raceId', 'driverId', 'stop', 'lap', 'milliseconds']].copy()

pit['under60'] = pit['milliseconds'] < 60_000
race_median = pit[pit['under60']].groupby('raceId')['milliseconds'].median()
pit['race_median_ms'] = pit['raceId'].map(race_median)
pit['valid'] = pit['under60'] & (pit['milliseconds'] <= 1.5 * pit['race_median_ms'])

print('原始進站筆數 :', len(pit))
print('有效進站筆數 :', pit['valid'].sum())
print('排除筆數     :', (~pit['valid']).sum())

races_name = raw['races'].set_index('raceId')[['year', 'name']]
(pit[~pit['valid']].groupby('raceId').size().rename('排除筆數')
   .to_frame().join(races_name).sort_values('排除筆數', ascending=False).head(8))

原始進站筆數 : 11371
有效進站筆數 : 10624
排除筆數     : 747


,排除筆數,year,name
raceId,,,
1100,49,2023,Australian Grand Prix
967,36,2016,Brazilian Grand Prix
1072,36,2021,Saudi Arabian Grand Prix
1111,33,2023,Dutch Grand Prix
1062,29,2021,Hungarian Grand Prix
1053,29,2021,Emilia Romagna Grand Prix
1039,25,2020,Tuscan Grand Prix
914,22,2014,Japanese Grand Prix


## 步驟 3-2：每人每場的進站統計

先把有效進站彙總成「每位車手、每場比賽」一列：

- `n_stops`：該場有效進站次數。
- `pit_s`：該場有效進站的平均時間（秒）。

**重要的定義選擇**：

- **只有正式完賽的比賽（`position` 不為空）才計算進站次數**。早早退賽的車手進站次數會是 0 或 1，這反映的是退賽而不是換胎策略，會把平均拉低。退賽場次設為缺值，不計入前 5 場平均。
- 完賽但完全沒進站（例如 2021 比利時只跑了幾圈安全車就結束，資料集中沒有任何進站紀錄）時：
  - 如果該場比賽本身就沒有進站資料，設為缺值（2021 比利時 raceId 1063 就是這種情況）。
  - 如果該場有進站資料、只是這位車手沒進站，`n_stops = 0`。
- 2011 年以前沒有進站資料，一律為缺值。

In [7]:
pit_valid = pit[pit['valid']]
pit_agg = (pit_valid.groupby(['raceId', 'driverId'])
           .agg(n_valid=('milliseconds', 'size'), pit_ms=('milliseconds', 'mean'))
           .reset_index())
pit_agg.to_sql('pit_agg', con, index=False)

pd.DataFrame({'raceId': pit['raceId'].unique()}).to_sql('races_with_pit', con, index=False)

pit_race = pd.read_sql('''
    SELECT  b.raceId, b.driverId, b.date,
            CASE
              WHEN b.position IS NULL                              THEN NULL  -- 未完賽：不計
              WHEN b.raceId NOT IN (SELECT raceId FROM races_with_pit) THEN NULL  -- 該場無進站資料
              ELSE COALESCE(p.n_valid, 0)                                          -- 完賽：有效進站次數
            END AS n_stops,
            CASE WHEN b.position IS NULL THEN NULL
                 ELSE p.pit_ms / 1000.0 END AS pit_s
    FROM base b
    LEFT JOIN pit_agg p ON p.raceId = b.raceId AND p.driverId = b.driverId
''', con)
pit_race.to_sql('pit_race', con, index=False)

pit_race[pit_race['n_stops'].notna()].describe().loc[['count', 'mean', 'min', 'max'], ['n_stops', 'pit_s']].round(2)

,n_stops,pit_s
count,5030.00,4975.00
mean,1.94,24.23
min,0.00,15.10
max,6.00,44.75


## 步驟 3-3：前 5 場平均（`n_pit_stops`, `avg_pit_duration`）

這是整份 notebook 最關鍵的 SQL 寫法，後面的步驟 4、5、6 都用同一個模式：

```sql
AVG(x) OVER (PARTITION BY driverId
             ORDER BY date, raceId
             ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING)
```

逐句解釋：

- `PARTITION BY driverId`：每位車手各自計算，不會混到別人的比賽。
- `ORDER BY date, raceId`：按比賽日期排序。`raceId` 只是同一天的保險排序。
- `ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING`：取「往前數第 5 場到往前數第 1 場」，**不含當場**。這是避免資料洩漏的關鍵：預測某場比賽時，只能用那場之前已知的資訊。
- `AVG` 會自動忽略缺值。所以如果前 5 場裡有 2 場退賽（缺值），就用另外 3 場平均。

「前 5 場」指**該車手自己參加的前 5 場**，不是賽曆上的前 5 場。車手缺席的比賽不算進去。

車手生涯第一場比賽沒有任何前一場，會是缺值。

In [8]:
feat_pit = pd.read_sql(f'''
    SELECT raceId, driverId,
           AVG(n_stops) OVER w AS n_pit_stops,
           AVG(pit_s)   OVER w AS avg_pit_duration
    FROM pit_race
    WINDOW w AS (PARTITION BY driverId ORDER BY date, raceId
                 ROWS BETWEEN {WINDOW} PRECEDING AND 1 PRECEDING)
''', con)
feat_pit.to_sql('feat_pit', con, index=False)
feat_pit.dropna().describe().round(2)

,raceId,driverId,n_pit_stops,avg_pit_duration
count,5875.00,5875.00,5875.00,5875.00
mean,989.67,566.31,1.95,24.25
std,89.20,378.00,0.55,1.72
min,842.00,1.00,0.50,18.85
25%,907.00,20.00,1.60,23.06
50%,990.00,818.00,2.00,24.10
75%,1067.00,839.00,2.25,25.31
max,1144.00,861.00,4.50,34.46


## 步驟 4-1：清理單圈時間

**問題**：不同賽道的圈速差異很大（摩納哥約 75 秒、斯帕約 105 秒）。如果直接把前 5 場的秒數平均，數字主要反映「最近跑了哪些賽道」，而不是車手的駕駛風格。

**解法**：先在每一場比賽內做標準化，把每圈時間換算成「該場代表圈速的百分比」：

$$\text{lap\_pct} = \frac{\text{該圈時間}}{\text{該場乾淨圈的中位數}} \times 100$$

例如 `100.8` 代表比該場典型圈速慢 0.8%。這樣不同賽道就可以放在一起比較。

**排除不代表正常駕駛的圈**：

1. 第 1 圈：起步圈，所有人都比較慢。
2. 進站圈（in-lap）與出站圈（out-lap）：用 `pit_stops` 的 `lap` 欄位找出來。
3. 慢於中位數 7% 以上的圈：通常是安全車、虛擬安全車、事故後的慢圈。7% 參考 F1 的 107% 規則。

只處理 2009 年以後的比賽，因為 2011 年的前 5 場平均最多只會往回用到 2010 年。這樣可以省下大量運算時間（`lap_times` 有近 59 萬列）。

In [9]:
races_dt = raw['races'][['raceId', 'year', 'date']]
lap = raw['lap_times'][['raceId', 'driverId', 'lap', 'milliseconds']].merge(races_dt, on='raceId')
lap = lap[lap['year'] >= START_YEAR - 2].copy()
n0 = len(lap)

# 1) 排除第 1 圈
lap = lap[lap['lap'] > 1]

# 2) 排除進站圈和出站圈
pit_laps = raw['pit_stops'][['raceId', 'driverId', 'lap']]
inout = pd.concat([pit_laps, pit_laps.assign(lap=pit_laps['lap'] + 1)]).drop_duplicates()
lap = lap.merge(inout.assign(is_pit=True), on=['raceId', 'driverId', 'lap'], how='left')
lap = lap[lap['is_pit'].isna()].drop(columns='is_pit')

# 3) 以該場中位數標準化，再排除慢於 107% 的圈
lap['race_median_ms'] = lap.groupby('raceId')['milliseconds'].transform('median')
lap['lap_pct'] = lap['milliseconds'] / lap['race_median_ms'] * 100
lap = lap[lap['lap_pct'] <= 107]

print(f'{START_YEAR-2} 年以後的單圈紀錄: {n0:,} 筆 → 清理後: {len(lap):,} 筆（保留 {len(lap)/n0:.1%}）')
lap[['milliseconds', 'lap_pct']].describe().round(2)

2009 年以後的單圈紀錄: 359,994 筆 → 清理後: 308,439 筆（保留 85.7%）


,milliseconds,lap_pct
count,308439.00,308439.00
mean,91992.49,99.86
std,12384.92,2.19
min,55404.00,76.08
25%,82082.00,98.73
50%,91331.00,99.85
75%,101376.00,101.04
max,134349.00,107.00


## 步驟 4-2：每人每場的單圈統計，再取前 5 場平均

每位車手每場比賽算出：

- `lap_mean_race`：乾淨圈 `lap_pct` 的平均。越小代表越快。
- `lap_sd_race`：乾淨圈 `lap_pct` 的標準差。越小代表圈速越穩定。注意：即使駕駛完全穩定，標準差也會受到油量減少（越跑越快）和輪胎衰退影響。

品質門檻：該車手在該場至少要有 **5 圈**乾淨圈，否則該場為缺值（例如開賽不久就退賽，幾圈的資料不足以代表風格）。

接著用和步驟 3-3 完全相同的 window function 算前 5 場平均，得到 `lap_time_mean`、`lap_time_sd`。

⚠️ 這兩個欄位的單位是**百分比**（相對於該場中位數），不是秒。

In [10]:
lap_race = (lap.groupby(['raceId', 'driverId'])['lap_pct']
               .agg(n_clean='size', lap_mean_race='mean', lap_sd_race='std')
               .reset_index())
lap_race.loc[lap_race['n_clean'] < 5, ['lap_mean_race', 'lap_sd_race']] = np.nan
lap_race.to_sql('lap_race', con, index=False)

feat_lap = pd.read_sql(f'''
    SELECT b.raceId, b.driverId,
           AVG(l.lap_mean_race) OVER w AS lap_time_mean,
           AVG(l.lap_sd_race)   OVER w AS lap_time_sd
    FROM base b
    LEFT JOIN lap_race l ON l.raceId = b.raceId AND l.driverId = b.driverId
    WINDOW w AS (PARTITION BY b.driverId ORDER BY b.date, b.raceId
                 ROWS BETWEEN {WINDOW} PRECEDING AND 1 PRECEDING)
''', con)
feat_lap.to_sql('feat_lap', con, index=False)
feat_lap.dropna().describe().round(3)

,raceId,driverId,lap_time_mean,lap_time_sd
count,6684.000,6684.000,6684.000,6684.000
mean,899.990,512.244,99.916,1.459
std,267.692,393.333,1.192,0.430
min,2.000,1.000,95.449,0.317
25%,879.000,18.000,99.160,1.178
50%,969.000,815.000,99.927,1.350
75%,1056.250,832.000,100.538,1.590
max,1144.000,861.000,106.260,4.670


## 步驟 5：`drv_avg_finish_l5`（車手前 5 場平均名次）

**定義**：該車手前 5 場正賽 `positionOrder` 的平均值，不含當場。

為什麼用 `positionOrder` 而不是 `position`：`position` 在退賽時是空的，如果用它，退賽場次會被忽略，平均名次會被高估（看起來比實際好）。`positionOrder` 會把退賽者排在完賽者之後（例如 20 人中第 3 個退賽的人是第 18 名），這樣退賽也會被反映為較差的表現。

數值越小代表近期表現越好。

In [11]:
feat_drv = pd.read_sql(f'''
    SELECT raceId, driverId,
           AVG(positionOrder) OVER (PARTITION BY driverId ORDER BY date, raceId
                                    ROWS BETWEEN {WINDOW} PRECEDING AND 1 PRECEDING) AS drv_avg_finish_l5
    FROM base
''', con)
feat_drv.to_sql('feat_drv', con, index=False)
feat_drv.dropna().describe().round(2)

,raceId,driverId,drv_avg_finish_l5
count,25898.00,25898.00,25898.00
mean,548.48,273.61,12.64
std,315.17,282.36,5.51
min,1.00,1.00,1.00
25%,296.00,56.00,9.00
50%,523.00,163.00,12.40
75%,810.00,375.00,15.80
max,1144.00,861.00,39.00


## 步驟 6：`con_points_l5`（車隊前 5 場平均積分）

**定義**：該車隊前 5 場正賽「兩位車手積分合計」的平均，不含當場。用來控制賽車本身的性能。

做法分兩層（不能直接在車手層級做 window）：

1. 先把 `results` 按 `(raceId, constructorId)` 加總，得到每隊每場的總積分。
2. 在車隊層級以 `PARTITION BY constructorId` 算前 5 場平均。
3. 再用 `(raceId, constructorId)` 接回每位車手。

注意事項：

- **積分制度**：2010 年起冠軍 25 分，2011 年起算的前 5 場最早只回溯到 2010 年，所以都在同一制度下。不過 2014 年阿布達比為雙倍積分，2019–2023 年最快圈有額外 1 分，這些小差異沒有另外調整。
- **只算正賽積分**，不含衝刺賽積分（定義為「正賽」）。
- ⚠️ **車隊改名會中斷歷史**：Ergast 把改名後的車隊視為新的 `constructorId`（例如 Toro Rosso → AlphaTauri → RB、Force India → Racing Point → Aston Martin、Sauber → Alfa Romeo → Sauber）。改名後第一場的 `con_points_l5` 會是缺值。下面的 cell 會列出受影響的列數。

In [12]:
feat_con = pd.read_sql(f'''
    WITH team_race AS (                      -- 每隊每場一列：兩位車手積分加總
        SELECT raceId, constructorId, MIN(date) AS date, SUM(points) AS team_pts
        FROM base
        GROUP BY raceId, constructorId
    )
    SELECT raceId, constructorId,
           AVG(team_pts) OVER (PARTITION BY constructorId ORDER BY date, raceId
                               ROWS BETWEEN {WINDOW} PRECEDING AND 1 PRECEDING) AS con_points_l5
    FROM team_race
''', con)
feat_con.to_sql('feat_con', con, index=False)

# 2011 年以後 con_points_l5 為缺值的車隊（通常是新車隊或改名後第一場）
chk = (base[base.year >= START_YEAR][['raceId', 'constructorId', 'year']].drop_duplicates()
       .merge(feat_con, on=['raceId', 'constructorId']))
miss = chk[chk['con_points_l5'].isna()].merge(raw['constructors'][['constructorId', 'name']])
print('2011 年以後 con_points_l5 缺值的車隊場次:')
miss[['year', 'raceId', 'constructorId', 'name']]

2011 年以後 con_points_l5 缺值的車隊場次:


,year,raceId,constructorId,name
0,2012,860,208,Lotus F1
1,2012,860,206,Marussia
2,2012,860,207,Caterham
3,2015,927,209,Manor Marussia
4,2016,948,210,Haas F1 Team
5,2019,1010,211,Racing Point
6,2020,1031,213,AlphaTauri
7,2021,1052,214,Alpine F1 Team
8,2024,1121,215,RB F1 Team


## 步驟 7：`street_circuit`（街道賽標記）

`circuits.csv` 沒有賽道類型欄位，所以要人工標記。下面以 `circuitRef`（賽道代碼）列出 2011 年以後用過的賽道並標記。

標記原則：

- **街道賽（=1）**：賽道主要在平常開放給一般車輛的城市道路上（摩納哥、新加坡、巴庫、吉達、拉斯維加斯、瓦倫西亞）。
- **半街道賽**：部分路段是公共道路或公園道路，但有常設設施（墨爾本 Albert Park、蒙特婁 Gilles Villeneuve、索契、邁阿密）。這類賽道的歸類在文獻中不一致。

由參數 `INCLUDE_SEMI_STREET` 控制半街道賽要不要算成 1。**預設為 `False`**（只有嚴格的街道賽是 1）。如果你的研究定義不同，改這個參數或編輯兩個清單即可。

In [13]:
INCLUDE_SEMI_STREET = False   # 半街道賽是否算成街道賽

STREET      = {'monaco', 'marina_bay', 'baku', 'jeddah', 'vegas', 'valencia'}
SEMI_STREET = {'albert_park', 'villeneuve', 'sochi', 'miami'}

circ = raw['circuits'][['circuitId', 'circuitRef', 'name', 'country']].copy()
street_set = STREET | SEMI_STREET if INCLUDE_SEMI_STREET else STREET
circ['street_circuit'] = circ['circuitRef'].isin(street_set).astype(int)
circ['類型'] = np.select([circ['circuitRef'].isin(STREET), circ['circuitRef'].isin(SEMI_STREET)],
                         ['街道賽', '半街道賽'], '常設賽道')
circ[['circuitId', 'street_circuit']].to_sql('feat_circuit', con, index=False)

used = raw['races'].query('year >= @START_YEAR')['circuitId'].unique()
circ[circ['circuitId'].isin(used)].sort_values(['類型', 'name'])[['circuitId', 'circuitRef', 'name', 'country', '類型', 'street_circuit']]

,circuitId,circuitRef,name,country,類型,street_circuit
0,1,albert_park,Albert Park Grand Prix Circuit,Australia,半街道賽,0
6,7,villeneuve,Circuit Gilles Villeneuve,Canada,半街道賽,0
76,79,miami,Miami International Autodrome,USA,半街道賽,0
70,71,sochi,Sochi Autodrom,Russia,半街道賽,0
20,21,imola,Autodromo Enzo e Dino Ferrari,Italy,常設賽道,0
73,76,mugello,Autodromo Internazionale del Mugello,Italy,常設賽道,0
13,14,monza,Autodromo Nazionale di Monza,Italy,常設賽道,0
31,32,rodriguez,Autódromo Hermanos Rodríguez,Mexico,常設賽道,0
72,75,portimao,Autódromo Internacional do Algarve,Portugal,常設賽道,0
17,18,interlagos,Autódromo José Carlos Pace,Brazil,常設賽道,0


## 步驟 8-1：天氣資料的時間對齊

天氣來自 **Open-Meteo Historical Weather API**（免費、免 API key，資料為 ERA5 再分析資料）。端點：`https://archive-api.open-meteo.com/v1/archive`

**取比賽時段，而不是全天**：研究問題是比賽中的天氣狀況，全天平均會混到深夜和清晨的氣溫，也會算進比賽前後的降雨。所以取「開賽時刻起約 2 小時」的逐時資料。

**時區陷阱**：`races.time` 是 UTC 時間，但 `races.date` 大多是當地日期，而且資料集本身不一致。拉斯維加斯是當地週六晚上 10 點開賽、UTC 已經是週日早上 6 點：

- 2023 年的 `date` 記成 UTC 日期（2023-11-19），`date + time` 剛好正確。
- 2024 年的 `date` 記成當地日期（2024-11-23），`date + time` 會早一天（變成當地週五晚上）。

所以不能只靠「日期 + 時間」，也不能一律加一天。

`race_utc_start()` 的修正方法利用「正賽一定在當地週六或週日舉行」：

1. 先取 `date + time` 當作 UTC 開賽時間。
2. 用經度粗估當地時間（經度 ÷ 15 小時），檢查當地是不是週六或週日。
3. 如果不是，依序嘗試往後一天、往前一天，取第一個落在週末的時間。

下面的輸出列出兩場拉斯維加斯和一場新加坡夜賽，可以確認修正結果，並列出所有被調整過的比賽（只有 2024 拉斯維加斯一場）。

In [14]:
def race_utc_start(date, time_utc, lng):
    '''回傳正確的 UTC 開賽時間：選擇讓「當地時間」落在週六或週日的那一天'''
    first = pd.Timestamp(f'{date} {time_utc}')
    for shift in (0, 1, -1):                                   # 優先不調整，再試 +1 天、-1 天
        utc = first + pd.Timedelta(days=shift)
        local_est = utc + pd.Timedelta(hours=lng / 15)         # 用經度粗估的當地時間
        if local_est.dayofweek >= 5:                           # 5 = 週六, 6 = 週日
            return utc
    return first

wx_races = (raw['races'].query('year >= @START_YEAR')[['raceId', 'year', 'name', 'date', 'time', 'circuitId']]
            .merge(raw['circuits'][['circuitId', 'lat', 'lng']], on='circuitId'))
wx_races['utc_start'] = [race_utc_start(d, t, g) for d, t, g in zip(wx_races['date'], wx_races['time'], wx_races['lng'])]
wx_races['local_est'] = wx_races['utc_start'] + pd.to_timedelta(wx_races['lng'] / 15, unit='h')
wx_races['adjusted']  = wx_races['utc_start'].dt.normalize() != pd.to_datetime(wx_races['date'])

print('需要查詢天氣的比賽數:', len(wx_races), '| 缺開賽時間:', wx_races['time'].isna().sum())
print('被調整日期的比賽:', wx_races.loc[wx_races['adjusted'], ['raceId', 'name', 'date']].values.tolist())
show = wx_races[wx_races['name'].str.contains('Las Vegas')].tail(2)
show = pd.concat([show, wx_races[wx_races['name'].str.contains('Singapore')].tail(1)])
show[['raceId', 'name', 'date', 'time', 'utc_start', 'local_est', 'adjusted']]

需要查詢天氣的比賽數: 286 | 缺開賽時間: 0
被調整日期的比賽: [[1142, 'Las Vegas Grand Prix', '2024-11-23']]


,raceId,name,date,time,utc_start,local_est,adjusted
260,1119,Las Vegas Grand Prix,2023-11-19,06:00:00,2023-11-19 06:00:00,2023-11-18 22:19:18.480000000,False
283,1142,Las Vegas Grand Prix,2024-11-23,06:00:00,2024-11-24 06:00:00,2024-11-23 22:19:18.480000000,True
279,1138,Singapore Grand Prix,2024-09-22,12:00:00,2024-09-22 12:00:00,2024-09-22 18:55:27.360000001,False


## 步驟 8-2：呼叫 Open-Meteo API

對每場比賽呼叫一次 API（2011 年以後約 286 場），取開賽日前後的逐時 `temperature_2m`（離地 2 公尺氣溫，°C）和 `precipitation`（降水量，mm），時區設為 GMT 以便和 `utc_start` 對齊。

欄位的計算：

- `air_temp_mean`：開賽時刻 h、h+1、h+2 三個整點氣溫的平均（h 為開賽時間取整點）。
- `precip_mm`：h+1、h+2 兩個整點的降水量加總。Open-Meteo 的逐時降水量代表「前一小時的累積量」，所以 h+1 的值就是開賽後第一個小時的雨量。

設計細節：

- **快取**：結果存到 `weather_cache.csv`。重跑 notebook 時直接讀快取，不會重複呼叫 API。如果中途失敗，下次只會補抓還沒成功的比賽。
- **連線失敗處理**：連續 3 次失敗就停止並印出提示（通常代表沒有網路或被防火牆擋住），天氣欄位留缺值，notebook 其他部分照常執行。
- 每次呼叫間隔 0.2 秒，避免太頻繁。

⚠️ ERA5 是約 9–25 公里網格的再分析資料，對局部短暫陣雨可能抓不準，論文中應列為限制。

In [15]:
WX_CACHE = OUT_DIR / 'weather_cache.csv'
API_URL  = 'https://archive-api.open-meteo.com/v1/archive'

def fetch_weather(row):
    '''查詢一場比賽的比賽時段天氣；回傳 (air_temp_mean, precip_mm)'''
    start = row.utc_start.floor('h')
    r = requests.get(API_URL, timeout=30, params={
        'latitude': row.lat, 'longitude': row.lng,
        'start_date': start.strftime('%Y-%m-%d'),
        'end_date':   (start + pd.Timedelta(days=1)).strftime('%Y-%m-%d'),
        'hourly': 'temperature_2m,precipitation',
        'timezone': 'GMT',
    })
    r.raise_for_status()
    h = pd.DataFrame(r.json()['hourly'])
    h['time'] = pd.to_datetime(h['time'])
    temp_hours   = [start + pd.Timedelta(hours=k) for k in (0, 1, 2)]
    precip_hours = [start + pd.Timedelta(hours=k) for k in (1, 2)]
    return (h.loc[h['time'].isin(temp_hours), 'temperature_2m'].mean(),
            h.loc[h['time'].isin(precip_hours), 'precipitation'].sum(min_count=1))

cache = pd.read_csv(WX_CACHE) if WX_CACHE.exists() else pd.DataFrame(columns=['raceId', 'air_temp_mean', 'precip_mm'])
todo = wx_races[~wx_races['raceId'].isin(cache['raceId'])]
print(f'快取中已有 {len(cache)} 場，這次要查詢 {len(todo)} 場')

new_rows, fails = [], 0
for row in todo.itertuples():
    try:
        t, p = fetch_weather(row)
        new_rows.append({'raceId': row.raceId, 'air_temp_mean': t, 'precip_mm': p})
        fails = 0
        time.sleep(0.2)
    except Exception as e:
        fails += 1
        print(f'raceId {row.raceId} 查詢失敗: {type(e).__name__}')
        if fails >= 3:
            print('連續 3 次失敗，停止查詢。請確認網路可以連到 archive-api.open-meteo.com 後重跑這個 cell。')
            break

if new_rows:
    cache = pd.concat([cache, pd.DataFrame(new_rows)], ignore_index=True)
    cache.to_csv(WX_CACHE, index=False)

cache = cache.astype({'raceId': int, 'air_temp_mean': float, 'precip_mm': float})
feat_wx = wx_races[['raceId']].merge(cache, on='raceId', how='left')
feat_wx.to_sql('feat_wx', con, index=False, if_exists='replace')
print(f'有天氣資料的比賽: {feat_wx["air_temp_mean"].notna().sum()} / {len(feat_wx)}')

快取中已有 0 場，這次要查詢 286 場
raceId 841 查詢失敗: ProxyError
raceId 842 查詢失敗: ProxyError
raceId 843 查詢失敗: ProxyError
連續 3 次失敗，停止查詢。請確認網路可以連到 archive-api.open-meteo.com 後重跑這個 cell。
有天氣資料的比賽: 0 / 286


## 步驟 9：合併成大表

以主表 `base` 為準，把步驟 2–8 的特徵表全部用 `LEFT JOIN` 接上，最後才篩出 2011 年以後。

- 用 `LEFT JOIN` 是為了**保證主表的列不會因為 join 而消失**。某個特徵表沒有對應資料時，該欄位為缺值。
- 每個特徵表的 key 都是唯一的（每人每場一列，或每隊每場一列，或每場一列），所以 join 也不會讓列數變多。下一個步驟會驗證這一點。
- `style_cluster` 是你第一階段分群的結果，這裡先放空值（`NULL`）佔位，欄位順序和研究設計一致。之後把分群結果用 `(raceId, driverId)` 接上即可。

In [16]:
master = pd.read_sql(f'''
    SELECT  b.raceId, b.year, b.circuitId, b.driverId, b.constructorId, b.grid,
            fq.quali_gap_pct,
            fp.n_pit_stops, fp.avg_pit_duration,
            fl.lap_time_mean, fl.lap_time_sd,
            fd.drv_avg_finish_l5,
            fc.con_points_l5,
            ci.street_circuit,
            wx.air_temp_mean, wx.precip_mm,
            NULL AS style_cluster,
            b.podium
    FROM base b
    LEFT JOIN feat_quali   fq ON fq.raceId = b.raceId AND fq.driverId = b.driverId
    LEFT JOIN feat_pit     fp ON fp.raceId = b.raceId AND fp.driverId = b.driverId
    LEFT JOIN feat_lap     fl ON fl.raceId = b.raceId AND fl.driverId = b.driverId
    LEFT JOIN feat_drv     fd ON fd.raceId = b.raceId AND fd.driverId = b.driverId
    LEFT JOIN feat_con     fc ON fc.raceId = b.raceId AND fc.constructorId = b.constructorId
    LEFT JOIN feat_circuit ci ON ci.circuitId = b.circuitId
    LEFT JOIN feat_wx      wx ON wx.raceId = b.raceId
    WHERE b.year >= {START_YEAR}
    ORDER BY b.date, b.raceId, b.positionOrder
''', con)

# 全部為空值的欄位，SQL 會回傳成文字型態；統一轉成浮點數，之後才能直接計算
for c in ['air_temp_mean', 'precip_mm', 'style_cluster']:
    master[c] = master[c].astype(float)

print(master.shape)
master.head(10)

(5980, 18)


,raceId,year,circuitId,driverId,constructorId,grid,quali_gap_pct,n_pit_stops,avg_pit_duration,lap_time_mean,lap_time_sd,drv_avg_finish_l5,con_points_l5,street_circuit,air_temp_mean,precip_mm,style_cluster,podium
0,841,2011,1,20,9,1,0.000000,NaN,NaN,96.401903,1.965629,4.4,29.6,0,NaN,NaN,NaN,1
1,841,2011,1,1,1,2,0.931413,NaN,NaN,97.218036,2.018976,6.4,21.4,0,NaN,NaN,NaN,1
2,841,2011,1,808,4,6,2.056771,NaN,NaN,98.780694,2.322936,14.6,7.2,0,NaN,NaN,NaN,1
3,841,2011,1,4,6,5,1.729938,NaN,NaN,96.638104,2.176760,3.0,21.2,0,NaN,NaN,NaN,0
4,841,2011,1,17,9,3,1.036766,NaN,NaN,98.223776,1.464666,7.6,29.6,0,NaN,NaN,NaN,0
5,841,2011,1,18,1,4,1.496486,NaN,NaN,97.444253,2.150066,5.6,21.4,0,NaN,NaN,NaN,0
6,841,2011,1,13,6,8,2.478181,NaN,NaN,97.612164,1.960382,11.4,21.2,0,NaN,NaN,NaN,0
7,841,2011,1,67,5,10,2.816986,NaN,NaN,99.110853,2.028472,14.4,0.6,0,NaN,NaN,NaN,0
8,841,2011,1,16,10,16,3.251565,NaN,NaN,98.510720,1.880627,13.6,2.0,0,NaN,NaN,NaN,0
9,841,2011,1,814,10,14,3.842977,NaN,NaN,NaN,NaN,NaN,2.0,0,NaN,NaN,NaN,0


## 步驟 10-1：品質檢查——列數、主鍵、目標變數

三個檢查：

1. **列數**：大表列數必須等於主表 2011 年以後的列數。如果變多，代表某個 join 的 key 不唯一，資料被重複複製了。
2. **主鍵唯一**：`(raceId, driverId)` 不能重複。
3. **目標變數合理性**：每場比賽應該剛好有 3 個 `podium = 1`，整體比例約為 3 ÷ 每場車手數（約 15%）。

In [17]:
n_base = (base['year'] >= START_YEAR).sum()
assert len(master) == n_base, '列數不一致：某個 join 造成重複'
assert not master.duplicated(['raceId', 'driverId']).any(), '主鍵重複'

pod_per_race = master.groupby('raceId')['podium'].sum()
print('✓ 列數與主表一致:', len(master))
print('✓ (raceId, driverId) 無重複')
print('比賽場數:', master['raceId'].nunique(), '| 年份範圍:', master['year'].min(), '–', master['year'].max())
print('每場 podium 數分佈:', pod_per_race.value_counts().to_dict())
print(f'podium 比例: {master["podium"].mean():.3f}')

✓ 列數與主表一致: 5980
✓ (raceId, driverId) 無重複
比賽場數: 286 | 年份範圍: 2011 – 2024
每場 podium 數分佈: {3: 286}
podium 比例: 0.143


## 步驟 10-2：品質檢查——資料洩漏

以一位車手（driverId = 1，Lewis Hamilton）2011 年的頭兩場比賽為例，手動驗證 `drv_avg_finish_l5`：

- 列出他在該場之前的 5 場比賽名次。
- 手算平均，和大表裡的值比對。

如果兩者相同，就證明前 5 場平均確實**不含當場**，而且是按時間順序取的。

In [18]:
hist = base[base['driverId'] == 1].sort_values(['date', 'raceId']).reset_index(drop=True)
for rid in master.loc[master['driverId'] == 1, 'raceId'].head(2):
    i = hist.index[hist['raceId'] == rid][0]
    prev = hist.iloc[i - WINDOW:i]
    manual = prev['positionOrder'].mean()
    in_table = master.loc[(master['raceId'] == rid) & (master['driverId'] == 1), 'drv_avg_finish_l5'].item()
    print(f'raceId {rid}（{hist.loc[i, "date"]}）前 5 場名次: {prev["positionOrder"].tolist()}'
          f' → 手算 {manual:.2f}，大表 {in_table:.2f}', '✓' if np.isclose(manual, in_table) else '✗')

raceId 841（2011-03-27）前 5 場名次: [19, 5, 2, 4, 2] → 手算 6.40，大表 6.40 ✓
raceId 842（2011-04-10）前 5 場名次: [5, 2, 4, 2, 2] → 手算 3.00，大表 3.00 ✓


## 步驟 10-3：品質檢查——缺值與分佈

列出每個欄位的缺值數、缺值比例和基本統計量。各欄位缺值的主要原因：

| 欄位 | 主要缺值原因 |
|---|---|
| `quali_gap_pct` | 排位賽沒有做出有效圈速 |
| `n_pit_stops`, `avg_pit_duration` | **2011 年第一場比賽全部缺值**（前 5 場都在 2010 年，當時沒有進站資料）；另外是車手前 5 場全部未完賽，或是車手生涯第一場 |
| `lap_time_mean`, `lap_time_sd` | 前 5 場都沒有足夠的乾淨圈，或是生涯第一場 |
| `drv_avg_finish_l5` | 車手生涯第一場 |
| `con_points_l5` | 新車隊或車隊改名後第一場 |
| `air_temp_mean`, `precip_mm` | API 未查詢成功 |
| `style_cluster` | 尚未接上第一階段分群結果（預期全部為缺值） |

另外注意 `quali_gap_pct` 的最大值：有些場次的排位賽天氣在 Q1 和 Q3 之間改變，或車手故障後只跑出很慢的一圈，差距會到 10% 以上（見步驟 2 的說明）。這些是真實紀錄，沒有刪除。建模時可以改用 `QUALI_METHOD = 'q1'`，或對這個欄位截尾（winsorize）。下面第二個輸出列出差距最大的幾筆。

In [19]:
summary = pd.DataFrame({
    '缺值數':   master.isna().sum(),
    '缺值比例': master.isna().mean().round(3),
}).join(master.describe().T[['mean', 'std', 'min', 'max']].round(3))
print('quali_gap_pct > 10% 的筆數:', (master['quali_gap_pct'] > 10).sum())
summary

quali_gap_pct > 10% 的筆數: 98


,缺值數,缺值比例,mean,std,min,max
raceId,0,0.000,988.388,89.687,841.000,1144.000
year,0,0.000,2017.499,4.125,2011.000,2024.000
circuitId,0,0.000,25.239,25.004,1.000,80.000
driverId,0,0.000,566.244,377.997,1.000,862.000
constructorId,0,0.000,71.860,86.282,1.000,215.000
grid,0,0.000,10.743,6.172,0.000,24.000
quali_gap_pct,82,0.014,2.367,2.678,0.000,45.596
n_pit_stops,104,0.017,1.954,0.546,0.000,4.500
avg_pit_duration,105,0.018,24.251,1.722,18.846,34.462
lap_time_mean,54,0.009,99.883,1.136,95.449,105.431


In [20]:
# quali_gap_pct 最大的 5 筆（附比賽名稱，方便查證）
(master.nlargest(5, 'quali_gap_pct')[['raceId', 'year', 'driverId', 'grid', 'quali_gap_pct']]
       .merge(raw['races'][['raceId', 'name']], on='raceId'))

,raceId,year,driverId,grid,quali_gap_pct,name
0,1099,2023,858,20,45.595649,Saudi Arabian Grand Prix
1,958,2016,837,21,37.796536,Hungarian Grand Prix
2,958,2016,836,20,34.237479,Hungarian Grand Prix
3,958,2016,828,22,33.788532,Hungarian Grand Prix
4,1044,2020,849,0,31.407229,Turkish Grand Prix


## 步驟 11：輸出

把大表存成 `f1_master.csv`（UTF-8，不含索引欄）。缺值在 CSV 中是空白，用 `pd.read_csv` 讀回來會自動變成 `NaN`。

In [21]:
out_path = OUT_DIR / 'f1_master.csv'
master.to_csv(out_path, index=False)
print('已輸出:', out_path.resolve(), master.shape)

已輸出: /home/claude/f1_master.csv (5980, 18)


## 欄位定義總表

| 欄位 | 定義 | 單位／值域 | 來源 | 是否為賽前可知 |
|---|---|---|---|---|
| `raceId` | 比賽編號（Ergast） | 整數 | races | — |
| `year` | 賽季年份 | 2011–2024 | races | — |
| `circuitId` | 賽道編號 | 整數 | races | — |
| `driverId` | 車手編號 | 整數 | results | — |
| `constructorId` | 車隊編號 | 整數 | results | — |
| `grid` | 正賽起跑位置；`0` = 從維修區起跑 | 0–24 | results | ✅ |
| `quali_gap_pct` | 車手最佳排位圈（Q1/Q2/Q3 取最快）比該場最快排位圈慢的百分比（`QUALI_METHOD='q1'` 時改為 Q1 vs Q1 最快） | %，≥ 0 | qualifying | ✅ |
| `n_pit_stops` | 該車手前 5 場**完賽**正賽的平均有效進站次數（排除紅旗等異常停留） | 次 | pit_stops + results | ✅ |
| `avg_pit_duration` | 該車手前 5 場完賽正賽的有效進站平均時間（含維修區行駛時間） | 秒 | pit_stops + results | ✅ |
| `lap_time_mean` | 該車手前 5 場正賽的平均乾淨圈速，以「該場乾淨圈中位數」標準化；越小越快 | % of race median（100 = 該場中位數） | lap_times | ✅ |
| `lap_time_sd` | 該車手前 5 場正賽乾淨圈速的標準差平均；越小越穩定 | 百分點 | lap_times | ✅ |
| `drv_avg_finish_l5` | 該車手前 5 場正賽的平均最終排序（`positionOrder`，退賽排在完賽者之後） | 名次，越小越好 | results | ✅ |
| `con_points_l5` | 該車隊前 5 場正賽兩位車手合計積分的平均 | 分 | results | ✅ |
| `street_circuit` | 街道賽 = 1、其他 = 0（人工標記，預設不含半街道賽） | 0/1 | circuits + 人工 | ✅ |
| `air_temp_mean` | 開賽起約 2 小時（3 個整點）的平均氣溫 | °C | Open-Meteo | ✅（預報） |
| `precip_mm` | 開賽後前 2 小時的累積降水量 | mm | Open-Meteo | ✅（預報） |
| `style_cluster` | 第一階段分群得到的駕駛風格群標籤（目前為空，待接入） | 類別 | 分群模型 | — |
| `podium` | **目標變數**：最終排序前三名 = 1，否則 = 0 | 0/1 | results | ❌（結果） |

**「前 5 場」的統一定義**：該車手（或車隊）自己參加的前 5 場正賽，依比賽日期排序，**不含當場**；其中缺值的場次自動跳過（以剩下的場次平均）。

**主要限制**

1. 天氣使用 ERA5 再分析資料，空間解析度有限，局部陣雨可能抓不準。真實預測時只能用天氣預報，準確度會比再分析資料差。
2. `quali_gap_pct`（預設 `'best'`）混合了 Q1/Q2/Q3 的成績。賽道進化會讓 Q1 淘汰者的差距被高估，排位賽中途天氣改變時尤其嚴重。可改用 `QUALI_METHOD = 'q1'`。
3. `con_points_l5` 在車隊改名時會中斷（新的 constructorId）。
4. 進站時間包含維修區行駛時間，不同賽道的維修區長度不同，所以 `avg_pit_duration` 有部分反映的是近期跑過哪些賽道。
5. `style_cluster` 若在全部資料上分群，再拿來做預測模型，會有資料洩漏；分群模型應該只用訓練期的資料 fit。